In [1]:
import os

import chromadb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load the TMDB Movies dataset
movies = pd.read_csv('../Data/tmdb_5000_movies.csv')
credits = pd.read_csv('../Data/tmdb_5000_credits.csv')

# Let's take a peek at what we're working with
print(f"Total movies: {len(movies)}")
movies[['title', 'overview', 'genres', 'release_date']].head(2)

Total movies: 4803


,title,overview,genres,release_date
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",2009-12-10
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",2007-05-19


In [2]:
import json


# Extract genres from JSON string
def extract_genres(genres_json):
    try:
        genres_list = json.loads(genres_json.replace("'", '"'))
        return [genre['name'] for genre in genres_list]
    except:
        return []

# Apply extraction
movies['genres_list'] = movies['genres'].apply(extract_genres)

# Now create rich document representations with movie descriptions
def create_movie_documents(movies):
    """Create rich document representations of movies with descriptions."""
    documents = []
    
    for _, movie in movies.iterrows():
        movie_id = movie['id']
        title = movie['title']
        overview = movie['overview']
        
        # Extract year from release date if present
        year = ""
        if pd.notna(movie['release_date']) and len(movie['release_date']) >= 4:
            year = movie['release_date'][:4]  # Just the year
        
        genres = movie['genres_list']
        
        # Create document text - this will be embedded
        doc = f"Movie: {title}\n"
        if year:
            doc += f"Year: {year}\n"
        if genres and len(genres) > 0:
            doc += f"Genres: {', '.join(genres)}\n"
        if overview and pd.notna(overview):
            doc += f"Overview: {overview}\n"
        
        documents.append({
            'id': str(movie_id),
            'content': doc,
            'metadata': {
                'title': title,
                'year': year,
                'genres': "|".join(genres),
                'overview': overview if pd.notna(overview) else ""
            }
        })
    
    return documents

# Create our document collection
movie_documents = create_movie_documents(movies)
print(f"Created {len(movie_documents)} movie documents")
print(f"Sample document:")
print(movie_documents[1]['content'])

Created 4803 movie documents
Sample document:
Movie: Pirates of the Caribbean: At World's End
Year: 2007
Genres: Adventure, Fantasy, Action
Overview: Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.



In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

chroma_client = chromadb.Client()

# Re-running this cell should reset the collection instead of erroring on
# "Collection already exists" (the in-memory client persists across reruns
# within the same kernel session).
try:
    chroma_client.delete_collection(name="my_movies_index")
except Exception:
    pass

collection = chroma_client.create_collection(name = "my_movies_index")

batch_size = 1000
for i in range(0, len(movie_documents), batch_size):
    batch = movie_documents[i:i + batch_size]
    ids = [doc['id'] for doc in batch]
    contents = [doc['content'] for doc in batch]
    metadatas = [doc['metadata'] for doc in batch]
    
    # Create embeddings for the batch
    embeddings = model.encode(contents, show_progress_bar=True)
    
    # Add to Chroma collection
    collection.add(
        ids=ids,
        documents=contents,
        metadatas=metadatas,
        embeddings=embeddings.tolist()  # Convert to list for JSON serialization
    )
    
print(f"Added {collection.count()} documents to the Chroma collection.")    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Added 4803 documents to the Chroma collection.


In [4]:
test_query = "A movie about mission which seems to impossible"
query_embedding = model.encode(test_query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)




for i , (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"Result {i+1}:")
    print(f"Document: {doc}")
    print(f"Metadata: {metadata}")
    print("\n")

Result 1:
Document: Movie: Mission: Impossible II
Year: 2000
Genres: Adventure, Action, Thriller
Overview: With computer genius Luther Stickell at his side and a beautiful thief on his mind, agent Ethan Hunt races across Australia and Spain to stop a former IMF agent from unleashing a genetically engineered biological weapon called Chimera. This mission, should Hunt choose to accept it, plunges him into the center of an international crisis of terrifying magnitude.

Metadata: {'year': '2000', 'overview': 'With computer genius Luther Stickell at his side and a beautiful thief on his mind, agent Ethan Hunt races across Australia and Spain to stop a former IMF agent from unleashing a genetically engineered biological weapon called Chimera. This mission, should Hunt choose to accept it, plunges him into the center of an international crisis of terrifying magnitude.', 'title': 'Mission: Impossible II', 'genres': 'Adventure|Action|Thriller'}


Result 2:
Document: Movie: Mission: Impossible -

In [5]:
def query_expansion(query):
    """Expand the query to improve retrieval."""
    # Simple rule-based expansions
    expansions = [
        query,  # Original query
        f"Movies similar to {query}",
        f"Plot description: {query}",
        f"Movie themes and content about {query}"
    ]
    return expansions


query_expansion(test_query)

['A movie about mission which seems to impossible',
 'Movies similar to A movie about mission which seems to impossible',
 'Plot description: A movie about mission which seems to impossible',
 'Movie themes and content about A movie about mission which seems to impossible']

In [6]:
def retrieve_movie_info(query, n_results=5):
    """Retrieve relevant movie information for a query."""
    # Expand query for better recall
    expanded_queries = query_expansion(query)
    
    all_results = []
    for expanded_query in expanded_queries:
        # Generate embedding for the query
        query_embedding = model.encode(expanded_query).tolist()
        
        # Retrieve relevant documents
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        
        # Add to results list
        for doc, metadata, id in zip(results['documents'][0], 
                                    results['metadatas'][0],
                                    results['ids'][0]):
            all_results.append({
                'id': id,
                'document': doc,
                'metadata': metadata,
                'query': expanded_query
            })
    
    # Remove duplicates (same movie ID)
    unique_results = {}
    for result in all_results:
        if result['id'] not in unique_results:
            unique_results[result['id']] = result
    
    return list(unique_results.values())

In [7]:
# Quick sanity check of retrieval before wiring up generation
sample_results = retrieve_movie_info(test_query, n_results=3)
for r in sample_results:
    print(r['metadata']['title'])

Mission: Impossible II
Mission: Impossible - Rogue Nation
Mission: Impossible - Ghost Protocol
Mission: Impossible
Mission to Mars


In [8]:
# In a terminal:
# ollama serve
# ollama pull llama3.2

In [9]:
import ollama
from pydantic import BaseModel, Field, ValidationError

OLLAMA_MODEL = "gemma3:4b"  # must match what you pulled with `ollama pull`

class Content(BaseModel):
    title: str = Field(..., description="title of the movie")
    why_picked: str = Field(..., description="why did you pick it, in ONE short sentence (max ~20 words)")

class ContentList(BaseModel):
    recommendations: list[Content]

def generate_rag_response(query, context, model_name=OLLAMA_MODEL, max_retries=2):
    # Format the prompt with retrieved context
    prompt = f"""
    You are a knowledgeable movie recommendation system with deep understanding of film plots, themes, and content. 
    Use the following retrieved information about movies to answer the user's question.
    
    User Query: {query}
    
    Retrieved Movie Information:
    {context}
    
    Based on this information, provide a helpful response to the user's query.
    If the user is asking about themes, plot elements, or specific content, use the movie overviews to provide detailed insights.

    If the retrieved information doesn't contain relevant details to answer the question, acknowledge the limitations and provide general movie information or suggestions.

    Keep "why_picked" to ONE short sentence per movie (max ~20 words) — do not write paragraphs, and do not add any text outside the JSON.
    """

    messages = [{"role": "user", "content": prompt}]
    last_error = None

    for attempt in range(max_retries + 1):
        response = ollama.chat(
            model=model_name,
            messages=messages,
            format=ContentList.model_json_schema(),  # ask Ollama to constrain output to this schema
            options={
                "temperature": 0.2,      # small models ramble/loop less at low temperature
                "repeat_penalty": 1.3,   # discourages the "2015} , 2015} , 2015}" style loops
                "num_predict": 2048,     # backstop headroom in case responses run long despite the length instruction
            },
        )
        raw = response["message"]["content"]
        try:
            parsed = ContentList.model_validate_json(raw)
            return parsed.recommendations
        except ValidationError as e:
            last_error = e
            # Feed the error back and ask the model to correct itself
            messages = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": raw},
                {"role": "user", "content": f"That response was not valid JSON matching the schema ({e}). Reply again with ONLY the corrected valid JSON — no explanation, no text before or after it."},
            ]

    raise last_error

In [10]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_results(query, results, top_k=5):
    """Re-rank results based on relevance to query."""
    # Prepare pairs for re-ranking
    pairs = [(query, result['document']) for result in results]

    # Get scores
    scores = reranker.predict(pairs)

    # Add scores to results
    for i, result in enumerate(results):
        result['rerank_score'] = float(scores[i])

    # Sort by score and take top_k
    reranked_results = sorted(results, key=lambda x: x['rerank_score'], reverse=True)[:top_k]

    return reranked_results

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [11]:
def movie_rag_without_rerank(user_query, max_context_docs=5):
    """RAG pipeline using raw retrieval order — no reranking."""
    results = retrieve_movie_info(user_query)
    top_results = results[:max_context_docs]

    context = "\n\n".join([res['document'] for res in top_results])
    response = generate_rag_response(user_query, context)

    return response, top_results


def movie_rag_with_rerank(user_query, max_context_docs=5):
    """RAG pipeline using cross-encoder reranked results."""
    results = retrieve_movie_info(user_query)
    top_results = rerank_results(user_query, results, top_k=max_context_docs)

    context = "\n\n".join([res['document'] for res in top_results])
    response = generate_rag_response(user_query, context)

    return response, top_results

In [12]:
def compare_rerank(query, n_results=5):
    """Show retrieval order before vs. after cross-encoder reranking, as a table."""
    raw_results = retrieve_movie_info(query)

    without_rerank = raw_results[:n_results]
    with_rerank = rerank_results(query, raw_results, top_k=n_results)

    comparison = pd.DataFrame({
        "Rank": range(1, n_results + 1),
        "Without rerank": [r['metadata']['title'] for r in without_rerank],
        "With rerank": [r['metadata']['title'] for r in with_rerank],
        "Rerank score": [round(r['rerank_score'], 3) for r in with_rerank],
    }).set_index("Rank")

    print(f"Query: {query}")
    return comparison

compare_rerank(test_query)

Query: A movie about mission which seems to impossible


,Without rerank,With rerank,Rerank score
Rank,,,
1,Mission: Impossible II,Mission: Impossible II,6.564
2,Mission: Impossible - Rogue Nation,Mission: Impossible,6.298
3,Mission: Impossible - Ghost Protocol,Mission: Impossible - Rogue Nation,5.965
4,Mission: Impossible,Mission: Impossible III,5.911
5,Mission: Impossible III,Mission: Impossible - Ghost Protocol,5.324


In [13]:
from itertools import zip_longest

without_response, without_sources = movie_rag_without_rerank(test_query)
with_response, with_sources = movie_rag_with_rerank(test_query)

without_lines = [f"{rec.title} — {rec.why_picked}" for rec in without_response]
with_lines = [f"{rec.title} — {rec.why_picked}" for rec in with_response]

# The LLM can return a different number of recommendations each run, so pad
# the shorter column instead of assuming equal lengths.
rows = list(zip_longest(without_lines, with_lines, fillvalue=""))

recs = pd.DataFrame(rows, columns=["Without rerank", "With rerank"])
recs.index += 1
recs

,Without rerank,With rerank
1,Mission: Impossible II — This film features a ...,Mission: Impossible II — This film features a ...
2,Mission: Impossible - Rogue Nation — The movie...,Mission: Impossible - Rogue Nation — It presen...
3,"Mission: Impossible – Ghost Protocol”, “why_p...",Mission: Impossible III — Hunt’s mission to pr...
4,"Mission: Impossible III”, “why_picked”: “This...",


In [14]:
def hyde_retrieval(query, n_results=5, model_name=OLLAMA_MODEL, verbose=True):
    """Implement HyDE for better retrieval."""
    # Step 1: Generate a hypothetical document using a local Ollama model
    hyde_prompt = f"""
    You are a movie expert. Generate a detailed description of a movie that would perfectly answer this query: "{query}"
    Include details about plot, themes, genre, and style.
    """

    hyde_response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": hyde_prompt}],
        options={
            "temperature": 0.7,      # HyDE wants a plausible, varied description, not a terse one
            "repeat_penalty": 1.3,
            "num_predict": 512,
        },
    )

    hypothetical_document = hyde_response["message"]["content"]

    if verbose:
        print("=== HyDE hypothetical document ===")
        print(hypothetical_document)
        print("===================================\n")

    # Step 2: Embed this hypothetical document instead of the query
    hyde_embedding = model.encode(hypothetical_document).tolist()

    # Step 3: Use this embedding for retrieval
    results = collection.query(
        query_embeddings=[hyde_embedding],
        n_results=n_results
    )

    # Format results
    formatted_results = []
    for doc, metadata, id in zip(results['documents'][0], 
                                results['metadatas'][0],
                                results['ids'][0]):
        formatted_results.append({
            'id': id,
            'document': doc,
            'metadata': metadata,
            'query': query
        })

    return formatted_results

In [15]:
def movie_hyde_rag_without_rerank(user_query, max_context_docs=5, hyde_results=None):
    """HyDE RAG pipeline using raw retrieval order — no reranking."""
    results = hyde_results if hyde_results is not None else hyde_retrieval(user_query, n_results=15)
    top_results = results[:max_context_docs]

    context = "\n\n".join([res['document'] for res in top_results])
    response = generate_rag_response(user_query, context)

    return response, top_results


def movie_hyde_rag_with_rerank(user_query, max_context_docs=5, hyde_results=None):
    """HyDE RAG pipeline using cross-encoder reranked results."""
    results = hyde_results if hyde_results is not None else hyde_retrieval(user_query, n_results=15)
    top_results = rerank_results(user_query, results, top_k=max_context_docs)

    context = "\n\n".join([res['document'] for res in top_results])
    response = generate_rag_response(user_query, context)

    return response, top_results

In [16]:
# Generate the HyDE hypothetical document once and reuse it for both branches
# below, so the comparison isolates the effect of reranking rather than
# mixing in a different HyDE hallucination each run.
hyde_candidates = hyde_retrieval(test_query, n_results=15)

hwithout_response, hwithout_sources = movie_hyde_rag_without_rerank(test_query, hyde_results=hyde_candidates)
hwith_response, hwith_sources = movie_hyde_rag_with_rerank(test_query, hyde_results=hyde_candidates)

hwithout_lines = [f"{rec.title} — {rec.why_picked}" for rec in hwithout_response]
hwith_lines = [f"{rec.title} — {rec.why_picked}" for rec in hwith_response]

hyde_rows = list(zip_longest(hwithout_lines, hwith_lines, fillvalue=""))

hyde_recs = pd.DataFrame(hyde_rows, columns=["Without rerank", "With rerank"])
hyde_recs.index += 1
hyde_recs

=== HyDE hypothetical document ===
Okay, let’s craft the perfect film for someone asking “A Movie About A Mission That Seems Impossible.” I'm calling it **“Echo Bloom”**.

**Genre:**  Neo-Noir Thriller with elements of Sci-Fi & Psychological Drama – leaning heavily into atmosphere and suspense. 


**Logline**: In a rain-soaked, perpetually twilight city built atop the ruins of an ancient underwater civilization, a disgraced ex-marine haunted by his past must infiltrate a clandestine organization manipulating reality itself to rescue his kidnapped daughter before time - *and sanity* - run out.

---

**Plot (Detailed Breakdown):** 


The film opens in Veridia – a sprawling metropolis clinging precariously on the cliffs of what was once coastal California, now submerged and transformed into an eerie underwater labyrinth by centuries-old seismic activity.  Veridian society is rigidly stratified: The 'High Tide' elite live within gleaming towers powered by geothermal energy harvested from b

,Without rerank,With rerank
1,The Abyss — This film features a seemingly imp...,Tomorrowland — This film features a mission to...
2,Subconscious — It involves a time-traveling in...,Sanctum — The movie depicts a dangerous expedi...
3,The Ruins — A group undertakes an impossible m...,


In [37]:
class Query(BaseModel):
    query: str

class QueryList(BaseModel):
    sub_queries: list[Query]
    
    
def decompose_query(query):
    """Decompose complex query into simpler sub-queries."""
    decompose_prompt = f"""
    Break down this complex movie-related query into 2-3 simpler sub-queries:
    "{query}"
    """
    
    decompose_response =  ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": decompose_prompt}],
            format=QueryList.model_json_schema(),  # ask Ollama to constrain output to this schema
            options={
                "temperature": 0.2,      # small models ramble/loop less at low temperature
                "repeat_penalty": 1.3,   # discourages the "2015} , 2015} , 2015}" style loops
                "num_predict": 2048,     # backstop headroom in case responses run long despite the length instruction
            },
        )

    
    # Parse the response to get sub-queries
    parsed = QueryList.model_validate_json(decompose_response.message.content)
    sub_queries = [q.query.strip() for q in parsed.sub_queries]
        
    return sub_queries


In [38]:
def retrieve_with_decomposition(query, n_results=3):
    """Retrieve using query decomposition for complex queries."""
    print(f"Original query: {query}")
    
    # Step 1: Decompose the query
    sub_queries = decompose_query(query)
    print(f"Decomposed into: {sub_queries}")
    
    # Step 2: Retrieve for each sub-query
    all_results = []
    for sub_query in sub_queries:
        # Get results for this sub-query
        sub_results = retrieve_movie_info(sub_query, n_results=n_results)
        print(f"Retrieved {len(sub_results)} results for: '{sub_query}'")
        all_results.extend(sub_results)
    
    # Step 3: Remove duplicates and sort by relevance
    unique_results = {}
    for result in all_results:
        if result['id'] not in unique_results:
            unique_results[result['id']] = result
    
    return list(unique_results.values())

def movie_decomposition_rag(user_query):
    # Retrieve relevant movie information
    results = retrieve_with_decomposition(user_query, 5)
    
    # Format context from retrieved documents
    context = "\n\n".join([res['document'] for res in results])

    # Generate response
    response = generate_rag_response(user_query, context)
    
    return response, results

In [39]:
complex_query = "I want science fiction movies that deal with time travel but also have strong character development and emotional depth"
decompose_query(complex_query)

['Find all Science Fiction Movies involving Time Travel.',
 'For the identified science fiction movies about time travel, filter for those with strong character development (e.g., complex characters, significant arcs).',
 'Finally, from that filtered list of films, select only those also exhibiting emotional depth – meaning they explore themes related to feelings and relationships.']

In [27]:
decompose_prompt = f"""
Break down this complex movie-related query into 2-3 simpler sub-queries:
"{complex_query}"
"""

decompose_response =  ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": decompose_prompt}],
        format=ContentList.model_json_schema(),  # ask Ollama to constrain output to this schema
        options={
            "temperature": 0.2,      # small models ramble/loop less at low temperature
            "repeat_penalty": 1.3,   # discourages the "2015} , 2015} , 2015}" style loops
            "num_predict": 2048,     # backstop headroom in case responses run long despite the length instruction
        },
    )


# Parse the response to get sub-queries

In [40]:
retrieve_with_decomposition(complex_query, n_results=3)

Original query: I want science fiction movies that deal with time travel but also have strong character development and emotional depth
Decomposed into: ['Find all Science Fiction Movies involving Time Travel.', 'Filter the results from above to include movies with Strong Character Development.']
Retrieved 4 results for: 'Find all Science Fiction Movies involving Time Travel.'
Retrieved 3 results for: 'Filter the results from above to include movies with Strong Character Development.'


[{'id': '14139',
  'document': 'Movie: Timecrimes\nYear: 2007\nGenres: Science Fiction, Thriller\nOverview: A man accidentally gets into a time machine and travels back in time nearly an hour. Finding himself will be the first of a series of disasters of unforeseeable consequences.\n',
  'metadata': {'genres': 'Science Fiction|Thriller',
   'overview': 'A man accidentally gets into a time machine and travels back in time nearly an hour. Finding himself will be the first of a series of disasters of unforeseeable consequences.',
   'year': '2007',
   'title': 'Timecrimes'},
  'query': 'Find all Science Fiction Movies involving Time Travel.'},
 {'id': '2135',
  'document': 'Movie: The Time Machine\nYear: 2002\nGenres: Science Fiction, Adventure, Action\nOverview: Hoping to alter the events of the past, a 19th century inventor instead travels 800,000 years into the future, where he finds humankind divided into two warring races.\n',
  'metadata': {'title': 'The Time Machine',
   'overview'

In [41]:
movie_decomposition_rag(complex_query)

Original query: I want science fiction movies that deal with time travel but also have strong character development and emotional depth
Decomposed into: ['Find all Science Fiction Movies involving Time Travel.', 'For the identified science fiction movies about time travel, filter for those with strong character development.', 'Finally, from that filtered list of films, select only those also exhibiting emotional depth']
Retrieved 8 results for: 'Find all Science Fiction Movies involving Time Travel.'
Retrieved 7 results for: 'For the identified science fiction movies about time travel, filter for those with strong character development.'
Retrieved 10 results for: 'Finally, from that filtered list of films, select only those also exhibiting emotional depth'


([Content(title='About Time', why_picked='This film excels in character development with a poignant story about family relationships using time travel as its core mechanic.'),
  Content(title='Back to the Future Part II', why_picked='It offers engaging plot twists and explores familial dynamics through multiple temporal journeys.” \xa0} , \xa0 { \x80\xa0“title”: “Timecrimes”, \x80\xa0”why_picked”: “This thriller uses time travel to create a complex, character-driven mystery with significant consequences.'),
  Content(title='The Time Machine', why_picked='It presents an epic future scenario and explores human evolution through the lens of extended temporal displacement.” \xa0} , {“title”: “Project Almanac”, ”why_picked”:'),
  Content(title='Back to the Future Part III', why_picked='This installment features a strong emotional core centered around Doc’s romance, alongside time travel adventures.'),
  Content(title='In Time', why_picked="It tackles social inequality and human connection w